In [1]:
import pandas as pd
import numpy as np 

In [2]:
# import os
# import time
# import json
# import requests
# import pandas as pd
# from tqdm import tqdm
# from pathlib import Path
# from concurrent.futures import ThreadPoolExecutor, as_completed

# # ==================== CONFIGURATION ====================
# TMDB_API_KEY = "28ed9fa013cb76618abf444748fd601d"  # <-- REPLACE WITH YOUR REAL KEY
# BASE_URL = "https://api.themoviedb.org/3"
# IMAGE_BASE_URL = "https://image.tmdb.org/t/p/original"
# HEADERS = {"Accept": "application/json"}

# TARGET = 6000                # total movies to collect
# SAVE_EVERY = 1000            # checkpoint every 1000 movies
# MAX_WORKERS = 17             # parallel detail requests
# REQUEST_TIMEOUT = 17         # seconds per request
# PAGE_DELAY = 0.1             # seconds between discover pages
# VOTE_THRESHOLD = 200         # movies with less than 200 votes will be excluded
# DATE_START = "2000-01-01"    # earliest release date
# DATE_END = "2026-12-31"      # latest release date

# OUT_DIR = Path("data")
# OUT_DIR.mkdir(exist_ok=True)
# CSV_PATH = OUT_DIR / "tmdb_movies.csv"

# # ==================== API KEY TEST ====================
# def test_api_key():
#     try:
#         r = requests.get(f"{BASE_URL}/movie/550", params={"api_key": TMDB_API_KEY}, timeout=10)
#         if r.status_code == 200:
#             print("✅ API key works!")
#             return True
#         else:
#             print(f"❌ Invalid API key: {r.json().get('status_message', 'Unknown error')}")
#             return False
#     except Exception as e:
#         print(f"❌ Network error: {e}")
#         return False

# # ==================== SAFE REQUEST ====================
# def tmdb_get(endpoint, params=None, retries=2):
#     params = params or {}
#     params["api_key"] = TMDB_API_KEY
#     url = f"{BASE_URL}/{endpoint}"
#     for attempt in range(retries):
#         try:
#             r = requests.get(url, params=params, headers=HEADERS, timeout=REQUEST_TIMEOUT)
#             if r.status_code == 200:
#                 return r.json()
#             if r.status_code == 429:
#                 time.sleep((attempt + 1) * 1.5)
#             else:
#                 return None
#         except Exception:
#             time.sleep(1)
#     return None

# # ==================== FETCH DETAILS (ONLY MAIN FIELDS) ====================
# def fetch_details(movie_id):
#     data = tmdb_get(f"movie/{movie_id}", params={"append_to_response": "keywords"})
#     if not data:
#         return None
    
#     # Extract keywords
#     keywords = [kw["name"] for kw in data.get("keywords", {}).get("keywords", [])]
    
#     # Poster URL
#     poster_path = data.get("poster_path")
#     poster_url = f"{IMAGE_BASE_URL}{poster_path}" if poster_path else None
    
#     return {
#         "id": data["id"],
#         "original_language": data.get("original_language"),
#         "overview": data.get("overview"),
#         "runtime": data.get("runtime"),
#         "tagline": data.get("tagline"),
#         "title": data.get("title"),
#         "genre_names": json.dumps([g["name"] for g in data.get("genres", [])]),
#         "movie_text": "",                                 # will be filled later
#         "poster_url": poster_url,
#         "keywords_names": json.dumps(keywords),          # renamed to match your spec
#     }

# # ==================== COLLECT FROM A SPECIFIC LANGUAGE ====================
# def collect_from_language(language_code, target, collected_ids, rows_buffer, start_page):
#     """
#     Collect movies for a given original language.
#     Returns: (new_collected, new_rows_buffer, last_page_used)
#     """
#     page = start_page
#     local_collected = 0
#     local_buffer = []
    
#     # Make a copy of the set to track local IDs
#     local_collected_ids = collected_ids.copy()
    
#     # Progress bar for this language
#     lang_pbar = tqdm(desc=f"Collecting {language_code.upper()}", unit="page", leave=False)
    
#     while local_collected < target and page <= 500:
#         params = {
#             "sort_by": "popularity.desc",
#             "primary_release_date.gte": DATE_START,
#             "primary_release_date.lte": DATE_END,
#             "vote_count.gte": VOTE_THRESHOLD,
#             "with_original_language": language_code,
#             "page": page,
#         }
#         resp = tmdb_get("discover/movie", params)
#         if not resp or not resp.get("results"):
#             break
        
#         new_ids = [m["id"] for m in resp["results"] if m["id"] not in local_collected_ids]
#         if new_ids:
#             with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
#                 futures = [executor.submit(fetch_details, mid) for mid in new_ids]
#                 for future in as_completed(futures):
#                     detail = future.result()
#                     if detail:
#                         local_buffer.append(detail)
#                         local_collected_ids.add(detail["id"])
#                         local_collected += 1
#                         rows_buffer.append(detail)    # add to global buffer
#                         collected_ids.add(detail["id"])
                    
#                     if local_collected >= target:
#                         break
        
#         lang_pbar.update(1)
#         lang_pbar.set_postfix({"collected": local_collected})
#         page += 1
#         time.sleep(PAGE_DELAY)
        
#         if local_collected >= target:
#             break
    
#     lang_pbar.close()
#     return local_collected, rows_buffer, page

# # ==================== MAIN ====================
# def main():
#     if not test_api_key():
#         return

#     # Load already collected IDs to resume
#     collected_ids = set()
#     if CSV_PATH.exists():
#         try:
#             existing = pd.read_csv(CSV_PATH)
#             collected_ids = set(existing["id"].astype(int))
#             print(f"Resuming from {len(collected_ids)} existing movies")
#         except:
#             print("Starting fresh")

#     total_collected = len(collected_ids)
#     rows_buffer = []
#     target_per_language = TARGET // 2 + 1  # about half each
    
#     print(f"Collecting {TARGET} high-quality movies from {DATE_START} to {DATE_END}...")
#     print(f"Minimum vote count: {VOTE_THRESHOLD}")
#     print()
    
#     # Collect English movies
#     print("📽️  Collecting English movies...")
#     en_collected, rows_buffer, _ = collect_from_language(
#         "en", target_per_language, collected_ids, rows_buffer, start_page=1
#     )
#     total_collected = len(collected_ids)
#     print(f"✅ Collected {en_collected} English movies (total: {total_collected})")
#     print()
    
#     # Save checkpoint after English collection
#     if rows_buffer:
#         df = pd.DataFrame(rows_buffer)
#         df.to_csv(CSV_PATH, mode='a', header=not CSV_PATH.exists(), index=False)
#         rows_buffer = []
#         print(f"💾 Saved checkpoint: {total_collected} movies")
#         print()
    
#     # Collect Hindi movies (Bollywood)
#     if total_collected < TARGET:
#         print("🎬  Collecting Hindi movies (Bollywood)...")
#         hi_collected, rows_buffer, _ = collect_from_language(
#             "hi", target_per_language, collected_ids, rows_buffer, start_page=1
#         )
#         total_collected = len(collected_ids)
#         print(f"✅ Collected {hi_collected} Hindi movies (total: {total_collected})")
    
#     # Final save
#     if rows_buffer:
#         df = pd.DataFrame(rows_buffer)
#         df.to_csv(CSV_PATH, mode='a', header=not CSV_PATH.exists(), index=False)
#         rows_buffer = []
    
#     print()
#     print(f"✅ DONE! Collected {total_collected} movies")
#     print(f"  - English: {total_collected - hi_collected if 'hi_collected' in locals() else en_collected}")
#     if 'hi_collected' in locals():
#         print(f"  - Hindi: {hi_collected}")
#     print(f"File saved at: {CSV_PATH.resolve()}")

# if __name__ == "__main__":
#     main()

In [3]:
df = pd.read_csv('/media/prince/5A4E832F4E83034D/movie recommender/data/tmdb_movies.csv')

In [4]:
df.columns.tolist()

['id',
 'title',
 'release_date',
 'runtime',
 'budget',
 'revenue',
 'popularity',
 'vote_average',
 'vote_count',
 'genres_names',
 'overview',
 'top_cast',
 'directors',
 'keywords_names',
 'movie_text']

In [ ]:
# import time
# import requests
# import pandas as pd
# from tqdm import tqdm
# from concurrent.futures import ThreadPoolExecutor, as_completed

# # ========== 1. SET YOUR REAL API KEY ==========
# TMDB_API_KEY = "28ed9fa013cb76618abf444748fd601d"   # <-- CHANGE THIS
# BASE_URL = "https://api.themoviedb.org/3"
# IMAGE_BASE_URL = "https://image.tmdb.org/t/p/original"
# MAX_WORKERS = 18

# # ========== 2. FETCH POSTER FOR A SINGLE MOVIE ID ==========
# def fetch_poster_url(movie_id):
#     url = f"{BASE_URL}/movie/{movie_id}"
#     params = {"api_key": TMDB_API_KEY}
#     try:
#         r = requests.get(url, params=params, timeout=10)
#         r.raise_for_status()
#         poster_path = r.json().get('poster_path')
#         return f"{IMAGE_BASE_URL}{poster_path}" if poster_path else None
#     except:
#         return None

# # ========== 3. ADD POSTER_URL TO YOUR EXISTING DF ==========
# # (Assumes your DataFrame is named 'df' and has an 'id' column)
# if 'poster_url' not in df.columns:
#     df['poster_url'] = None

# # Find movies that still need a poster
# missing_ids = df[df['poster_url'].isna()]['id'].tolist()
# print(f"Fetching posters for {len(missing_ids)} movies...")

# if missing_ids:
#     poster_map = {}
#     with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
#         futures = {ex.submit(fetch_poster_url, mid): mid for mid in missing_ids}
#         for fut in tqdm(as_completed(futures), total=len(missing_ids), desc="Posters"):
#             mid = futures[fut]
#             try:
#                 poster_map[mid] = fut.result()
#             except:
#                 poster_map[mid] = None

#     # Update the column
#     df['poster_url'] = df['id'].map(poster_map).fillna(df['poster_url'])

#     # Optional: save the updated DataFrame
#     df.to_csv("movies_with_posters.csv", index=False)
#     print("✅ Poster URLs added and saved to 'movies_with_posters.csv'")
# else:
#     print("All movies already have a poster URL.")

Fetching posters for 6000 movies...


Posters: 100%|██████████| 6000/6000 [52:42<00:00,  1.90it/s]   


✅ Poster URLs added and saved to 'movies_with_posters.csv'


In [6]:
df2 = pd.read_csv('/media/prince/5A4E832F4E83034D/movie recommender/movies_with_posters.csv')

In [9]:
df2['row_index'] = range(len(df2))

In [10]:
df2['row_index']

0          0
1          1
2          2
3          3
4          4
        ... 
5995    5995
5996    5996
5997    5997
5998    5998
5999    5999
Name: row_index, Length: 6000, dtype: int64

In [12]:
df2.to_csv('tmdb_with_index.csv', index=False)